# IT3091 - Member 4 - Weekly wholesale price model



In [1]:
from pathlib import Path
import os, subprocess
REPO_URL = "https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
REPO_PATH = Path("/content/IT3091---Machine-Learning-Assignment")
if not REPO_PATH.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
os.chdir(REPO_PATH)
print("Repository:", REPO_PATH)

Repository: /content/IT3091---Machine-Learning-Assignment


## Remove old monthly outputs safely
Only known generated monthly artifacts are removed; raw data and weekly files are kept.

In [2]:
from pathlib import Path
old_monthly = [
    REPO_PATH / "03_IT24101090_JAYAKODY_ND_Price_EDA_Preprocessing.ipynb",
    REPO_PATH / "04_IT23361690_Gunathilake_BMP_Final_Model.ipynb",
    REPO_PATH / "Preprocessed Datasets/03_final_modelling_dataset.csv",
    REPO_PATH / "Preprocessed Datasets/04_final_modelling_dataset.csv",
    REPO_PATH / "final_test_metrics.csv",
    REPO_PATH / "final_test_predictions.csv",
    REPO_PATH / "final_multioutput_model.joblib",
    REPO_PATH / "final_recommendation_example.json",
    REPO_PATH / "model_metadata.json",
    REPO_PATH / "recommendation_limitations.csv",
]
removed = []
for path in old_monthly:
    if path.exists():
        path.unlink()
        removed.append(str(path.relative_to(REPO_PATH)))
print("Removed old monthly notebooks and outputs:", removed if removed else "none found")
print("Weekly files are preserved.")

Removed old monthly notebooks and outputs: ['03_IT24101090_JAYAKODY_ND_Price_EDA_Preprocessing.ipynb', '04_IT23361690_Gunathilake_BMP_Final_Model.ipynb', 'Preprocessed Datasets/03_final_modelling_dataset.csv', 'Preprocessed Datasets/04_final_modelling_dataset.csv', 'final_test_metrics.csv', 'final_test_predictions.csv', 'final_multioutput_model.joblib', 'final_recommendation_example.json', 'model_metadata.json', 'recommendation_limitations.csv']
Weekly files are preserved.


In [6]:
import re
import pandas as pd

raw_files = list(
    (REPO_PATH / "Raw Datasets").glob(
        "*Weekly*Average*Retail*Prices*.xlsx"
    )
)

if not raw_files:
    raise FileNotFoundError(
        "Weekly price workbook not found."
    )

price_file = raw_files[0]

preview = pd.read_excel(
    price_file,
    header=None,
    nrows=20
)

header_row = next(
    i for i, row in preview.iterrows()
    if row.astype(str)
    .str.contains("Market Type", case=False)
    .any()
)

raw = pd.read_excel(
    price_file,
    header=header_row
)

raw.columns = [
    str(column).strip()
    for column in raw.columns
]

market_col = next(
    column for column in raw.columns
    if column.lower() == "market type"
)

commodity_col = next(
    column for column in raw.columns
    if column.lower() in {"commodity", "crop"}
)

year_col = next(
    column for column in raw.columns
    if column.lower() == "year"
)

week_cols = [
    column for column in raw.columns
    if re.fullmatch(
        r"W\s*\d{1,2}",
        str(column),
        flags=re.IGNORECASE
    )
]

raw = raw[
    raw[market_col]
    .astype(str)
    .str.contains("wholesale", case=False, na=False)
]

long = raw[
    [commodity_col, year_col] + week_cols
].rename(
    columns={
        commodity_col: "Crop",
        year_col: "Year"
    }
).melt(
    id_vars=["Crop", "Year"],
    var_name="Week",
    value_name="WholesalePrice_RsKg"
)

long["Year"] = pd.to_numeric(
    long["Year"],
    errors="coerce"
)

long["WeekNumber"] = pd.to_numeric(
    long["Week"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)

long["WholesalePrice_RsKg"] = pd.to_numeric(
    long["WholesalePrice_RsKg"]
    .astype(str)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

long = long.dropna(
    subset=[
        "Crop",
        "Year",
        "WeekNumber",
        "WholesalePrice_RsKg"
    ]
).copy()

long["Year"] = long["Year"].astype(int)
long["WeekNumber"] = long["WeekNumber"].astype(int)

long["WeekDate"] = pd.to_datetime(
    long["Year"].astype(str)
    + "-W"
    + long["WeekNumber"].astype(str)
    + "-1",
    format="%G-W%V-%u",
    errors="coerce"
)

weekly = (
    long
    .groupby(
        ["Crop", "WeekDate"],
        as_index=False
    )["WholesalePrice_RsKg"]
    .mean()
    .sort_values(["Crop", "WeekDate"])
)

grouped = weekly.groupby(
    "Crop",
    group_keys=False
)

weekly["PreviousWeekWholesalePrice_RsKg"] = (
    grouped["WholesalePrice_RsKg"].shift(1)
)

weekly["NextWeekWholesalePrice_RsKg"] = (
    grouped["WholesalePrice_RsKg"].shift(-1)
)

weekly["Year"] = weekly["WeekDate"].dt.year
weekly["WeekNumber"] = (
    weekly["WeekDate"]
    .dt.isocalendar()
    .week
    .astype(int)
)
weekly["Month"] = weekly["WeekDate"].dt.month
weekly["Market"] = "Colombo and suburbs"

weekly = weekly[
    [
        "WeekDate",
        "Crop",
        "Market",
        "Year",
        "WeekNumber",
        "Month",
        "WholesalePrice_RsKg",
        "PreviousWeekWholesalePrice_RsKg",
        "NextWeekWholesalePrice_RsKg"
    ]
]

output_file = (
    REPO_PATH
    / "Preprocessed Datasets"
    / "03_price_preprocessed.csv"
)

output_file.parent.mkdir(
    parents=True,
    exist_ok=True
)

weekly.to_csv(
    output_file,
    index=False
)

print("Weekly records created:", len(weekly))
print("Saved:", output_file)
display(weekly.head())

Weekly records created: 15598
Saved: /content/IT3091---Machine-Learning-Assignment/Preprocessed Datasets/03_price_preprocessed.csv


,WeekDate,Crop,Market,Year,WeekNumber,Month,WholesalePrice_RsKg,PreviousWeekWholesalePrice_RsKg,NextWeekWholesalePrice_RsKg
0,2012-01-02,ASH PLANTAINS,Colombo and suburbs,2012,1,1,41.00,NaN,41.11
1,2012-01-09,ASH PLANTAINS,Colombo and suburbs,2012,2,1,41.11,41.00,44.00
2,2012-01-16,ASH PLANTAINS,Colombo and suburbs,2012,3,1,44.00,41.11,44.50
3,2012-01-23,ASH PLANTAINS,Colombo and suburbs,2012,4,1,44.50,44.00,36.25
4,2012-01-30,ASH PLANTAINS,Colombo and suburbs,2012,5,1,36.25,44.50,40.00


## Train and evaluate the weekly model

In [7]:
!pip -q install joblib
import json, warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")
weekly_file = REPO_PATH / "Preprocessed Datasets/03_price_preprocessed.csv"
if not weekly_file.exists():
    raise FileNotFoundError("Run notebook 03 first.")
weekly = pd.read_csv(weekly_file, parse_dates=["WeekDate"])
required = ["Crop","Market","Year","WeekNumber","Month","WholesalePrice_RsKg","PreviousWeekWholesalePrice_RsKg","NextWeekWholesalePrice_RsKg"]
missing = [c for c in required if c not in weekly.columns]
if missing:
    raise ValueError("Missing weekly columns: " + str(missing) + ". Re-run notebook 03.")
weekly = weekly.dropna(subset=required).copy()
features = ["Crop","Market","Year","WeekNumber","Month","WholesalePrice_RsKg","PreviousWeekWholesalePrice_RsKg"]
target = "NextWeekWholesalePrice_RsKg"
train, test = weekly[weekly.Year <= 2019], weekly[weekly.Year >= 2020]
cat, num = ["Crop","Market"], ["Year","WeekNumber","Month","WholesalePrice_RsKg","PreviousWeekWholesalePrice_RsKg"]
def make_pipe(estimator):
    prep = ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),cat),("num",StandardScaler(),num)])
    return Pipeline([("preprocessor",prep),("model",estimator)])
estimators = {"Dummy baseline":DummyRegressor(strategy="mean"),"Linear regression":LinearRegression(),"Random forest":RandomForestRegressor(n_estimators=200,min_samples_leaf=2,random_state=42,n_jobs=-1),"Gradient boosting":GradientBoostingRegressor(n_estimators=150,max_depth=3,loss="huber",random_state=42)}
rows, fitted = [], {}
for name, estimator in estimators.items():
    pipe = make_pipe(estimator).fit(train[features], train[target])
    pred = pipe.predict(test[features])
    rows.append({"Model":name,"Records":len(weekly),"TrainRecords":len(train),"TestRecords":len(test),"MAE":mean_absolute_error(test[target],pred),"RMSE":np.sqrt(mean_squared_error(test[target],pred)),"R2":r2_score(test[target],pred)})
    fitted[name] = (pipe, pred)
comparison = pd.DataFrame(rows).sort_values(["R2","MAE"], ascending=[False,True]).reset_index(drop=True)
best_name = comparison.loc[0,"Model"]
best_model, best_pred = fitted[best_name]
comparison.to_csv(REPO_PATH/"weekly_price_model_comparison.csv", index=False)
comparison.iloc[[0]].to_csv(REPO_PATH/"weekly_price_test_metrics.csv", index=False)
predictions = test[["WeekDate","Crop","Market",target]].copy()
predictions["Predicted_NextWeekWholesalePrice_RsKg"] = best_pred
predictions.to_csv(REPO_PATH/"weekly_price_test_predictions.csv", index=False)
joblib.dump(best_model, REPO_PATH/"weekly_price_model.joblib")
(REPO_PATH/"weekly_price_model_metadata.json").write_text(json.dumps({"model":best_name,"target":target,"frequency":"weekly","horizon":"next week"},indent=2), encoding="utf-8")
print("Selected model:", best_name)
print("Weekly records used:", len(weekly))
display(comparison.round(4))

Selected model: Linear regression
Weekly records used: 15546


,Model,Records,TrainRecords,TestRecords,MAE,RMSE,R2
0,Linear regression,15546,9387,6159,27.2432,48.3255,0.9060
1,Random forest,15546,9387,6159,31.6688,72.4445,0.7887
2,Gradient boosting,15546,9387,6159,32.4833,80.0892,0.7417
3,Dummy baseline,15546,9387,6159,118.8673,190.1336,-0.4558


In [ ]:
from getpass import getpass
import json
import os, subprocess
from pathlib import Path
try:
    from google.colab import _message
    current_notebook = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
    notebook_path = REPO_PATH / "04_IT23361690_Gunathilake_BMP_Weekly_Only.ipynb"
    notebook_path.write_text(json.dumps(current_notebook["ipynb"], ensure_ascii=False, indent=1), encoding="utf-8")
except Exception as exc:
    print("Notebook auto-save was skipped:", exc)
github_username = input("GitHub username: ").strip()
github_email = input("GitHub email: ").strip()
github_token = getpass("GitHub Personal Access Token: ").strip()
subprocess.run(["git", "config", "user.name", github_username], cwd=REPO_PATH, check=True)
subprocess.run(["git", "config", "user.email", github_email], cwd=REPO_PATH, check=True)
files = ["04_IT23361690_Gunathilake_BMP_Weekly_Only.ipynb", "weekly_price_model_comparison.csv", "weekly_price_test_metrics.csv", "weekly_price_test_predictions.csv", "weekly_price_model.joblib", "weekly_price_model_metadata.json"]
subprocess.run(["git", "add", "-A"], cwd=REPO_PATH, check=True)
subprocess.run(["git", "commit", "-m", "Add weekly next-week price model"], cwd=REPO_PATH, check=False)
askpass = Path("/tmp/git_askpass.sh")
askpass.write_text("#!/bin/sh\ncase \"$1\" in\n  *Username*) echo \"$GITHUB_USERNAME\" ;;\n  *Password*) echo \"$GITHUB_TOKEN\" ;;\nesac\n", encoding="utf-8")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({"GITHUB_USERNAME": github_username, "GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass)})
result = subprocess.run(["git", "push", "origin", "main"], cwd=REPO_PATH, env=env, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Push failed. Check token permissions and repository access.")
print("Member 4 weekly contribution pushed successfully.")